# Task 4 — Graph Topology Ingestion into Neo4j

## Mục tiêu

Ghi node và edge trực tiếp từ Kafka Connect vào Neo4j, không qua Spark, đồng
thời giữ idempotency khi replay.

## Thiết kế và triển khai

Hai connector độc lập đọc `cpg.nodes` và `cpg.edges`. Cypher xử lý
UPSERT/DELETE; `MERGE` dùng `node_id` hoặc `edge_id`, còn DELETE dọn topology
của revision trước. Constraint unique được tạo trước khi đăng ký connector.
Source:
[`neo4j-sink-nodes.json`](https://github.com/linh285/huggingface-cpg-streaming/blob/main/task4/connectors/neo4j-sink-nodes.json),
[`neo4j-sink-edges.json`](https://github.com/linh285/huggingface-cpg-streaming/blob/main/task4/connectors/neo4j-sink-edges.json).

## Lệnh thực thi

```bash
bash task4/scripts/register_connectors.sh
python task4/verify_neo4j.py --connect-url "$KAFKA_CONNECT_URL" \
  --expected-nodes 208003 --expected-edges 267695 \
  --expected-ast-edges 207856 --expected-cfg-edges 18549 \
  --expected-dfg-edges 29557 --expected-call-edges 11733 \
  --require-zero-placeholders --timeout 300 \
  --json artifacts/task4/neo4j_corpus_verification.json
```

## Kết quả thực nghiệm


In [1]:
import json
from pathlib import Path

proof = json.loads(Path("../artifacts/task4/neo4j_corpus_verification.json").read_text(encoding="utf-8"))
counts = proof["counts"]
assert proof["status"] == "PASS"
assert counts == {
    "total_nodes": 208003,
    "total_edges": 267695,
    "distinct_node_ids": 208003,
    "distinct_edge_ids": 267695,
    "placeholder_nodes": 0,
}
assert proof["edge_breakdown"] == {
    "AST": 207856, "CFG": 18549, "DFG": 29557, "CALL": 11733
}
for status in proof["connectors"].values():
    assert status["connector"] == "RUNNING"
    assert status["tasks"] == ["RUNNING"]
print(json.dumps({
    "counts": counts,
    "edge_breakdown": proof["edge_breakdown"],
    "duplicate_node_ids": len(proof["duplicate_node_ids"]),
    "duplicate_edge_ids": len(proof["duplicate_edge_ids"]),
    "connectors": proof["connectors"],
}, indent=2))


{
  "counts": {
    "total_nodes": 208003,
    "total_edges": 267695,
    "distinct_node_ids": 208003,
    "distinct_edge_ids": 267695,
    "placeholder_nodes": 0
  },
  "edge_breakdown": {
    "AST": 207856,
    "DFG": 29557,
    "CFG": 18549,
    "CALL": 11733
  },
  "duplicate_node_ids": 0,
  "duplicate_edge_ids": 0,
  "connectors": {
    "neo4j-sink-cpg-nodes": {
      "connector": "RUNNING",
      "tasks": [
        "RUNNING"
      ]
    },
    "neo4j-sink-cpg-edges": {
      "connector": "RUNNING",
      "tasks": [
        "RUNNING"
      ]
    }
  }
}


![Neo4j Browser — full corpus counts](images/task4/neo4j-full-corpus-counts.jpg)

## Vấn đề và cách xử lý

Node và edge nằm ở hai topic nên edge có thể đến trước node; edge connector tạo
placeholder endpoint và node connector hoàn thiện thuộc tính sau đó. Khi publish
file lớn, Docker thiếu RAM làm API treo; heap/page cache được giới hạn và
consumer lag được đưa về 0 trước khi truy vấn.

## Đánh giá

Neo4j chứa 208.003 node và 267.695 edge, bằng đúng số ID distinct; placeholder
và duplicate đều bằng 0. Hệ thống hiện dùng một connector task cho mỗi topic,
phù hợp máy lab nhưng thông lượng phụ thuộc tài nguyên Neo4j.
